[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/04_sequence_qc/04_numbering_lab.ipynb)


# 04 — 입력 QC — numbering · CDR · germline · 번호 체계

> 본문 [`04_sequence_qc.md`](04_sequence_qc.md) 와 **한 절씩 짝지어** 보세요.
> **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행해요.
> 이 노트북의 표·그래프·수치는 **여러분이 방금 돌린 `my_run/`** 에서 나와요. 실행을 건너뛰거나 실패하면 저장소에 커밋된 `data/`(실제 실행 산출물) 로 자동 폴백해요.
>
> **실측 소요 —** ANARCI numbering+germline(H·L 동시) **0.15초** · abnumber CDR 추출 **1초 미만**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`04_sequence_qc`) 폴더로 이동한 뒤, 필요한 패키지만 깝니다.
- **로컬**: 이 노트북을 `04_sequence_qc/` 폴더 안에서 열었다면 클론 없이 그대로 진행돼요.

이 셀이 끝나면 두 개의 경로가 준비돼요 — **`my_run/`**(내가 직접 만들 결과)과 **`data/`**(저장소에 커밋된 레퍼런스 결과).
아래 랩은 항상 `my_run/` 을 먼저 찾고, 없으면 `data/` 로 폴백하면서 **어느 쪽을 쓰는지 print** 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌려요. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행돼요 — pip 만으로는 `hmmscan` 이 없어 죽어요.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "04_sequence_qc"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas anarci abnumber"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

# ANARCI 는 numbering 을 hmmscan(HMMER) 서브프로세스로 돌려요 — pip 만으로는 hmmscan 이 없어요.
if APT_PKGS and IN_COLAB:
    _run("apt-get -qq update")                 # 인덱스가 낡으면 install 이 404 로 죽는다
    _run(f"apt-get -qq install -y {APT_PKGS}")
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨져요.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — ANARCI numbering + germline (본문 4.1)

```bash
ANARCI -i parental.fasta --scheme imgt --assign_germline --use_species human --csv -o my_run/anarci_gl
```

`--assign_germline` 이 핵심이에요. 이게 있어야 "가장 가까운 사람 germline 과 몇 % 같은가" 가 나오고, 그 숫자가 humanization 전략을 정해요.

In [ ]:
t0 = time.time()
try:
    r = subprocess.run(["ANARCI", "-i", "data/parental.fasta", "--scheme", "imgt", "--csv",
                        "-o", str(MY / "anarci_gl"), "--assign_germline", "--use_species", "human"],
                       capture_output=True, text=True)
    rc, err = r.returncode, r.stderr
except FileNotFoundError as e:
    # ANARCI CLI 자체가 PATH 에 없음 (hmmscan 이 없으면 numbering 도 여기서 죽어요 — Ch.03)
    rc, err = 127, f"{e} — ANARCI/hmmscan 이 PATH 에 없어요"
el = time.time() - t0
if rc != 0:
    print("ANARCI CLI 실패:", str(err).strip()[:300])
    print("→ hmmscan 이 PATH 에 있는지 확인하세요(Ch.03). 아래 셀은 레퍼런스로 이어져요.")
else:
    print(f"ANARCI numbering+germline 완료: {el:.2f}초")
    for p in sorted(MY.glob("anarci_gl_*.csv")):
        print("  →", p)

## 2) 내 결과 확인 — germline 표 (본문 4.2)

V identity 가 낮은 체인 = 사람 germline 과 멀다 = **humanization 여지가 크다**.

In [ ]:
import pandas as pd, glob

def germline_table(paths):
    rows = []
    for p in paths:
        for _, r in pd.read_csv(p).iterrows():
            rows.append({"체인": r["chain_type"], "hmm_species": r["hmm_species"],
                         "V gene": r["v_gene"], "V identity": float(r["v_identity"]),
                         "J gene": r["j_gene"], "J identity": float(r["j_identity"])})
    return pd.DataFrame(rows)

mine_paths = sorted(MY.glob("anarci_gl_*.csv"))
if mine_paths:
    print("[내 결과]", *[str(p) for p in mine_paths])
    gl = germline_table(mine_paths)
else:
    print("[레퍼런스] data/anarci_imgt_H.csv · data/anarci_imgt_KL.csv")
    gl = germline_table(["data/anarci_imgt_H.csv", "data/anarci_imgt_KL.csv"])
display(gl)

print("해석 — heavy V identity 가 낮을수록 heavy framework 에 손댈 자리가 많아요.")
print("       light 가 이미 높으면 노력의 무게중심은 heavy 로 갑니다.")

## 3) 번호 체계와 J-gene 동점 — 두 개의 함정 (실측)

**(a) J-gene 은 동점이에요.** 본문 4.2 표는 heavy J 를 `IGHJ4*01` 86% 로 적었지만, ANARCI 실측은 **`IGHJ6*01` 85.71%** 입니다.
둘 다 J-segment 14 잔기 중 12개 일치(12/14 = 85.71%)로 **정확히 동점**이고, 어느 쪽을 고르냐는 **도구의 tie-break** 차이예요
(ANARCI 는 germline dict 순회에서 먼저 만난 `IGHJ6*01`, abnumber 0.3.2 의 별도 germline 조회는 `IGHJ4*01`).
아래 셀에서 **두 도구를 같은 서열에 돌려 동점을 직접 확인**합니다.

**(b) 번호 체계가 두 개예요.** 뒤 챕터에서 도구별 mutation 표기가 섞이에요.

| 체계 | 어디서 쓰나 | 예 |
|---|---|---|
| **raw 1-based** | Sapiens·AnthroAb 의 `predict_scores` 가 그대로 쓰는 서열 인덱스 | `I78T` = 입력 문자열의 78번째 잔기 |
| **IMGT** | ANARCI/abnumber numbering. gap·삽입이 있어 raw 와 어긋남 | `H86` = 같은 잔기의 IMGT 번호 |

Humatch 는 indel 을 만들 수 있어(우리 VL 은 1 잔기 짧아져요) **raw 인덱스 비교가 깨져요.**
그래서 도구 간 비교는 **반드시 IMGT 로 변환**해서 합니다. 그 변환표를 지금 만들어요.

In [ ]:
HAVE_ABNUMBER, ch_h, ch_l = _have("abnumber"), None, None

if HAVE_ABNUMBER:
    from abnumber import Chain

    # (a) 동점 확인 — 같은 서열, 두 도구
    ch_h = Chain(VH, scheme="imgt", assign_germline=True)
    anarci_j = gl.loc[gl["체인"] == "H", "J gene"].iloc[0] if (gl["체인"] == "H").any() else "?"
    print(f"ANARCI   J gene : {anarci_j}")
    print(f"abnumber J gene : {ch_h.j_gene}")
    print("→ 같은 서열인데 J 가 달라요. 12/14 = 85.71% 동점이라 tie-break 가 갈린 것이에요(오류 아님).")
    print("   본문 4.2 의 'IGHJ4*01 86%' 는 abnumber 쪽 tie-break 입니다.\n")

    # (b) raw ↔ IMGT 크로스워크 만들기
    def raw2imgt(seq):
        ch = Chain(seq, scheme="imgt")
        assert seq.startswith(ch.seq), "numbering 영역이 서열 앞부분과 어긋나요"
        m, i, last = {}, 1, None
        for pos, aa in ch:
            m[i] = str(pos); last = str(pos); i += 1
        for k, aa in enumerate(ch.tail, start=1):       # IMGT 범위 밖(C-말단 꼬리)
            m[i] = f"{last}_tail{k}"; i += 1
        return ch, m

    ch_h, r2i_H = raw2imgt(VH)
    ch_l, r2i_L = raw2imgt(VL)
    (MY / "raw2imgt_H.json").write_text(json.dumps({str(k): v for k, v in r2i_H.items()}, indent=1))
    (MY / "raw2imgt_L.json").write_text(json.dumps({str(k): v for k, v in r2i_L.items()}, indent=1))
    print("→", MY / "raw2imgt_H.json", "·", MY / "raw2imgt_L.json")
else:
    print("[레퍼런스] abnumber 가 없어 커밋된 크로스워크(data/raw2imgt_*.json)를 씁니다")
    r2i_H = {int(k): v for k, v in json.loads(pathlib.Path("data/raw2imgt_H.json").read_text()).items()}
    r2i_L = {int(k): v for k, v in json.loads(pathlib.Path("data/raw2imgt_L.json").read_text()).items()}

print("\nraw → IMGT 예시 (VH):", {k: r2i_H[k] for k in (5, 12, 78, 115)})
print("raw → IMGT 예시 (VL):", {k: r2i_L[k] for k in (31, 85, 109, 111)})
print("VL 마지막 잔기:", r2i_L[len(VL)], "← IMGT 범위 밖(C-말단 꼬리)이라 tail 로 라벨링돼요")
print("\n핵심 — Sapiens 의 'I78T' 는 raw 78번, 같은 잔기의 IMGT 번호는", r2i_H[78], "입니다.")

## 4) 직접 실행 — CDR 추출 + **보호 좌표 못 박기** (본문 4.3)

humanization 에서 가장 먼저 할 일은 "여기는 절대 안 건드린다"를 좌표로 고정하는 것이에요.
CDR 을 **raw 인덱스**(Sapiens/AnthroAb 가 쓰는 좌표)와 **IMGT 라벨** 두 가지로 모두 저장해요 — Ch.05 의 CDR 가드가 이 파일을 씁니다.

In [ ]:
if HAVE_ABNUMBER:
    seqs_by_chain = {"H": VH, "L": VL}
    cdr_seqs = [(tag, name, "".join(ch.regions[name].values()))
                for tag, ch in (("H", ch_h), ("L", ch_l))
                for name in ("CDR1", "CDR2", "CDR3")]
else:
    print("[레퍼런스] data/cdr_table_imgt.csv 의 CDR 서열로 좌표를 복원해요")
    seqs_by_chain = {"H": VH, "L": VL}
    cdr_seqs = [(r["chain"], r["cdr"], r["sequence"]) for _, r in pd.read_csv("data/cdr_table_imgt.csv").iterrows()]

cdr_rows, guard = [], {"H": [], "L": []}
for tag, name, s in cdr_seqs:
    seq = seqs_by_chain[tag]
    start = seq.find(s) + 1                            # raw 1-based 시작
    r2i = r2i_H if tag == "H" else r2i_L
    guard[tag] += list(range(start, start + len(s)))
    cdr_rows.append({"chain": tag, "cdr": name, "sequence": s,
                     "raw_start": start, "raw_end": start + len(s) - 1,
                     "imgt": f"{r2i[start]}..{r2i[start + len(s) - 1]}"})

cdr = pd.DataFrame(cdr_rows)
display(cdr)
print("보호 좌표(raw 1-based) — VH:", len(guard["H"]), "잔기 / VL:", len(guard["L"]), "잔기")
if HAVE_ABNUMBER:
    cdr.to_csv(MY / "cdr_table_imgt.csv", index=False)
    (MY / "cdr_guard.json").write_text(json.dumps(guard, indent=1))
    print("→", MY / "cdr_table_imgt.csv", "·", MY / "cdr_guard.json")
print("\nCDR-H3 =", cdr.loc[(cdr.chain=='H') & (cdr.cdr=='CDR3'), 'sequence'].iloc[0],
      "— 항원 결합에 가장 결정적인 loop. 여기 mutation 이 들어가면 빨간불이에요.")

## 5) 레퍼런스 대조 — 커밋된 실행 결과와 맞춰보기

`data/` 는 이 가이드를 만들 때 **실제로 돌려 나온** 산출물이에요. 내 결과와 한 글자씩 비교해요.

In [ ]:
ref_cdr = pd.read_csv("data/cdr_table_imgt.csv")
ref_gl  = pd.read_csv("data/germline_assignment.csv")
ref_r2i_H = json.loads(pathlib.Path("data/raw2imgt_H.json").read_text())

ok_cdr = all(ref_cdr.loc[(ref_cdr.chain == r.chain) & (ref_cdr.cdr == r.cdr), "sequence"].iloc[0] == r.sequence
             for r in cdr.itertuples())
ok_map = {str(k): v for k, v in r2i_H.items()} == ref_r2i_H
print("CDR 6개 일치      :", ok_cdr)
print("raw→IMGT(H) 일치  :", ok_map)

print("\n[레퍼런스 germline — ANARCI 1.3 실측]")
display(ref_gl[["chain", "gene_type", "gene", "identity_pct"]])
print("실측: VH IGHV1-69*06 63.27% / IGHJ6*01 85.71%  ·  VL IGLV1-40*01 80.85% / IGLJ2*01 83.33%")
print("본문 4.2 의 J 'IGHJ4*01 86%' 는 동점 tie-break 차이예요(3항 참고).")

## 이 랩에서 확인한 것

1. **germline 실측** — VH `IGHV1-69*06` **63.27%** / VL `IGLV1-40*01` **80.85%**. heavy 가 훨씬 비인간적 → 손댈 자리가 많아요.
2. **J-gene 은 동점** — `IGHJ6*01`(ANARCI) vs `IGHJ4*01`(abnumber) 둘 다 85.71%. 도구의 tie-break 차이지 오류가 아니에요.
3. **CDR 6개** — H `GYTFTDYV`/`IYPGSGTN`/`ARRGRYGLYAMDY`, L `SSDVGHKFP`/`KNL`/`QSYDSSLRVV`.
4. **번호 체계 두 개**(raw 1-based ↔ IMGT)를 명시적으로 이어 붙인 `raw2imgt_*.json` 을 만들었어요. Ch.06 의 도구 간 합의 분석이 이 파일 위에서 돕니다.
5. **CDR 보호 좌표**(`cdr_guard.json`) 를 못 박았어요 — Ch.05 에서 이걸로 사고를 막아요.

다음 → **[Ch.05 — Sapiens](../05_humanize_sapiens/05_sapiens_lab.ipynb)**